# Text Generation with RNN, LSTM, and Transformer

## (Simplified Version) to run in colab

This notebook covers:
1. **RNN** - Basic recurrent network
2. **LSTM** - Long Short-Term Memory
3. **Transformer** - Attention-based model
4. **GPT-2** - Pre-trained transformer

Complete the `# TODO:` sections.

Use Colab GPU and models directly from Pytorch instead of building yourself.

## Setup

In [ ]:
!pip install -q transformers

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import math
import time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using: {device}")

## Data Preparation

We use a small subset of text for fast training.

In [ ]:
# Small sample text for quick training
text = """To be or not to be that is the question.
Whether tis nobler in the mind to suffer the slings and arrows of outrageous fortune.
Or to take arms against a sea of troubles and by opposing end them.
To die to sleep no more and by a sleep to say we end the heartache.
The thousand natural shocks that flesh is heir to tis a consummation devoutly to be wished.
To die to sleep to sleep perchance to dream ay theres the rub.
For in that sleep of death what dreams may come when we have shuffled off this mortal coil.
Must give us pause theres the respect that makes calamity of so long life.
For who would bear the whips and scorns of time the oppressors wrong the proud mans contumely.
The pangs of despised love the laws delay the insolence of office.
And the spurns that patient merit of the unworthy takes.
""" * 20  # Repeat to get more data

print(f"Text length: {len(text)} characters")

###Build Vocabulary

In [ ]:
# Create vocabulary
# 1. Get sorted unique characters
# 2. Create char_to_idx mapping
# 3. Create idx_to_char mapping

chars = sorted(list(set(text)))
vocab_size = len(chars)
char_to_idx = {ch: i for i, ch in enumerate(chars)}
idx_to_char = {i: ch for i, ch in enumerate(chars)}

print(f"Vocab size: {vocab_size}")

### Create Dataset

In [ ]:
class TextDataset(Dataset):
    def __init__(self, text, char_to_idx, seq_len):
        self.seq_len = seq_len
        #  Encode text as list of indices
        self.data = [char_to_idx[c] for c in text]

    def __len__(self):
        # Return number of sequences
        return len(self.data) - self.seq_len

    def __getitem__(self, idx):
        # Return (input, target) tensors
        x = self.data[idx:idx+self.seq_len]
        y = self.data[idx+1:idx+self.seq_len+1]
        return torch.tensor(x), torch.tensor(y)

In [ ]:
# Hyperparameters (small for fast training)
SEQ_LEN = 32
BATCH_SIZE = 64
EMBED_DIM = 64
HIDDEN_DIM = 128
NUM_LAYERS = 2
EPOCHS = 50
LR = 0.003

# Create dataset and dataloader
dataset = TextDataset(text, char_to_idx, SEQ_LEN)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)
print(f"Batches per epoch: {len(loader)}")

---
## Part 1: RNN Model

### Exercise: Implement RNN

In [ ]:
class RNNModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers):
        super().__init__()
        # TODO: Define layers
        self.embedding = None  # TODO: nn.Embedding( )
        self.rnn = None  # TODO: nn.RNN( )
        self.fc = None  # TODO: nn.Linear( )

    def forward(self, x):
        # TODO: Implement forward pass
        # 1. Embedding
        # 2. RNN
        # 3. Linear layer
        x = None  # TODO
        x, _ = None  # TODO
        x = None  # TODO
        return x

### Exercise: Training Function

In [ ]:
def train(model, loader, epochs, lr):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    model.train()
    start_time=time.time()

    for epoch in range(epochs):
        total_loss = 0

        for x, y in loader:
            x, y = x.to(device), y.to(device)

            # TODO: Training step
            # 1. Zero gradients
            # 2. Forward pass
            # 3. Reshape output: (batch*seq, vocab) and target: (batch*seq)
            # 4. Calculate loss
            # 5. Backward
            # 6. Update weights

            optimizer.zero_grad()
            out = None  # TODO: model(x)
            loss = None  # TODO: criterion(out.view(-1, vocab_size), y.view(-1))
            # TODO: loss.backward()
            # TODO: optimizer.step()

            total_loss += loss.item()

        if (epoch + 1) % 5 == 0:
            avg_loss = total_loss / len(loader)
            print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}, Perplexity: {math.exp(avg_loss):.2f}; time: {(time.time() - start_time):.4f}s")
            start_time=time.time()

    return model

### Exercise: Text Generation

In [ ]:
def generate(model, seed, length=100, temperature=0.5):
    model.eval()
    chars = list(seed)

    with torch.no_grad():
        for _ in range(length):
            # TODO: Generate next character
            # 1. Convert current chars to tensor
            # 2. Forward pass
            # 3. Get last timestep logits, apply temperature
            # 4. Sample from softmax probabilities
            # 5. Append new character

            x = torch.tensor([[char_to_idx[c] for c in chars[-SEQ_LEN:]]]).to(device)
            out = None  # TODO: model(x)
            logits = None  # TODO: out[0, -1] / temperature
            probs = None  # TODO: torch.softmax(logits, dim=0)
            idx = None  # TODO: torch.multinomial(probs, 1).item()
            chars.append(idx_to_char[idx])

    return ''.join(chars)

In [ ]:
# Train RNN
print("Training RNN...")
rnn = RNNModel(vocab_size, EMBED_DIM, HIDDEN_DIM, NUM_LAYERS).to(device)
rnn = train(rnn, loader, EPOCHS, LR)
print("\nGenerated text:")
print(generate(rnn, "To be or"))

---
## Part 2: LSTM Model

### Exercise: Implement LSTM

In [ ]:
class LSTMModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers):
        super().__init__()
        # TODO: Define layers (similar to RNN but use nn.LSTM)
        self.embedding = None  # TODO
        self.lstm = None  # TODO: nn.LSTM()
        self.fc = None  # TODO

    def forward(self, x):
        # TODO: Implement forward pass
        # Note: LSTM returns (output, (hidden, cell))
        x = None  # TODO
        x, _ = None  # TODO
        x = None  # TODO
        return x

In [ ]:
# Train LSTM
print("Training LSTM...")
lstm = LSTMModel(vocab_size, EMBED_DIM, HIDDEN_DIM, NUM_LAYERS).to(device)
lstm = train(lstm, loader, EPOCHS, LR)
print("\nGenerated text:")
print(generate(lstm, "To be or"))

---
## Part 3: Transformer Model

### Exercise: Positional Encoding

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()
        # TODO: Create positional encoding
        # PE(pos, 2i) = sin(pos / 10000^(2i/d_model))
        # PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))

        pe[:, 0::2] = None  # TODO
        pe[:, 1::2] = None  # TODO

        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

### Exercise: Transformer Model

In [ ]:
class TransformerModel(nn.Module):
    def __init__(self, vocab_size, d_model, nhead, num_layers):
        super().__init__()
        self.d_model = d_model

        # TODO: Define layers
        self.embedding = None  # TODO: nn.Embedding(vocab_size, d_model)
        self.pos_enc = None  # TODO: PositionalEncoding(d_model)

        # Use PyTorch's TransformerEncoder for simplicity
        encoder_layer = nn.TransformerEncoderLayer(d_model, nhead, d_model*4, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers)

        self.fc = None  # TODO: nn.Linear(d_model, vocab_size)

    def forward(self, x):
        # TODO: Implement forward pass
        # 1. Embedding * sqrt(d_model)
        # 2. Add positional encoding
        # 3. Create causal mask
        # 4. Transformer
        # 5. Linear

        seq_len = x.size(1)

        # Causal mask (prevent attending to future tokens)
        mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool().to(x.device)

        x = None  # TODO
        x = None  # TODO
        x = None  # TODO
        x = None  # TODO
        return x

In [ ]:
# Train Transformer
print("Training Transformer...")
transformer = TransformerModel(vocab_size, d_model=64, nhead=4, num_layers=2).to(device)
transformer = train(transformer, loader, EPOCHS, LR)
print("\nGenerated text:")
print(generate(transformer, "To be or"))

---
## Part 4: Evaluation

In [ ]:
def evaluate_model(model, loader):
    """Calculate test loss and perplexity"""
    model.eval()
    criterion = nn.CrossEntropyLoss()
    total_loss = 0

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss = criterion(out.view(-1, vocab_size), y.view(-1))
            total_loss += loss.item()

    avg_loss = total_loss / len(loader)
    return avg_loss, math.exp(avg_loss)

# Compare models
print("\n" + "="*50)
print("MODEL COMPARISON")
print("="*50)

for name, model in [("RNN", rnn), ("LSTM", lstm), ("Transformer", transformer)]:
    loss, ppl = evaluate_model(model, loader)
    print(f"{name:12} | Loss: {loss:.4f} | Perplexity: {ppl:.2f}")

print("\n" + "="*50)
print("GENERATED SAMPLES")
print("="*50)

seed = "To die to"
for name, model in [("RNN", rnn), ("LSTM", lstm), ("Transformer", transformer)]:
    print(f"\n{name}:")
    print(generate(model, seed, length=80))

### Play

In [ ]:
# Generate from any text
def generate_from_input(seed_text, length=100, temperature=0.2):
    """Generate text from custom input using all models"""
    print(f"Seed: '{seed_text}'\n" + "="*50)

    for name, model in [("RNN", rnn), ("LSTM", lstm), ("Transformer", transformer)]:
        generated = generate(model, seed_text, length=length, temperature=temperature)
        print(f"\n{name}:\n{generated}")

while True:
    seed = input("Enter text (or 'quit'): ")
    if seed.lower() == 'quit':
        break
    generate_from_input(seed, length=80)

In [ ]:
def predict_next(model, text, top_k=5):
    """Show top-k predictions for the next character"""
    model.eval()
    with torch.no_grad():
        x = torch.tensor([[char_to_idx[c] for c in text[-SEQ_LEN:]]]).to(device)
        out = model(x)
        probs = torch.softmax(out[0, -1], dim=0)
        top_probs, top_idx = probs.topk(top_k)

        print(f"Input: '{text}'")
        print(f"Next character predictions:")
        for p, i in zip(top_probs, top_idx):
            print(f"  '{idx_to_char[i.item()]}' : {p.item()*100:.1f}%")

# Example
predict_next(rnn, "To be or not to b")
predict_next(lstm, "To be or not to b")
predict_next(transformer, "To be or not to b")

---
## Part 5: Pre-trained GPT-2

Now let's use a pre-trained transformer that's already learned from massive amounts of text.

### Exercise 9: Load and Use GPT-2

In [ ]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer

# TODO: Load pre-trained GPT-2
tokenizer = None  # TODO: GPT2Tokenizer.from_pretrained('gpt2')
gpt2 = None  # TODO: GPT2LMHeadModel.from_pretrained('gpt2')
gpt2 = gpt2.to(device)
gpt2.eval()

print(f"GPT-2 parameters: {sum(p.numel() for p in gpt2.parameters()):,}")

### Exercise 10: Generate with GPT-2

In [ ]:
def generate_gpt2(prompt, max_length=100, temperature=0.8):
    # TODO: Generate text with GPT-2
    # 1. Encode prompt
    # 2. Generate with model.generate()
    # 3. Decode output

    input_ids = None  # TODO: tokenizer.encode(prompt, return_tensors='pt').to(device)

    with torch.no_grad():
        output = None  # TODO: gpt2.generate( )

    return None  # TODO: tokenizer.decode()

In [ ]:
# Test GPT-2
prompts = [
    "To be or not to be",
    "The meaning of life is",
    "Artificial intelligence will"
]

print("="*60)
print("GPT-2 GENERATION")
print("="*60)

for prompt in prompts:
    print(f"\nPrompt: {prompt}")
    print("-"*40)
    print(generate_gpt2(prompt, max_length=60))